# Salud Chilecito - Demostracion del DAO

Notebook de demostracion para mostrar todas las operaciones de la capa DAO contra Oracle.

**Autor:** Alesandro David Fajardo / Kevin Facundo Nunez
**Universidad:** Universidad Nacional de Chilecito
**Ano:** 2026

## 1. Conexion a la base de datos

In [ ]:
from dao import SaludDAO

dao = SaludDAO()
print(f"Conexion: {dao.ping()}")

## 2. Tablas y estructura del esquema

In [ ]:
# Contar registros en cada tabla principal
tablas = ["especialidad", "centro_salud", "medico", "paciente", "agenda_medico", "turno", "historial_clinico"]

print("=== Registros por tabla ===")
for tabla in tablas:
    total = dao.contar(tabla)
    print(f"  {tabla:25s} -> {total} registros")

## 3. Especialidades

In [ ]:
especialidades = dao.listar_especialidades()

print("=== Especialidades ===")
for e in especialidades:
    print(f"  ID {e['id_especialidad']:2d} | {e['nombre']:20s} | {e['descripcion']}")

## 4. Centros de salud

In [ ]:
centros = dao.listar_centros()

print("=== Centros de Salud ===")
for c in centros:
    print(f"  ID {c['id_centro']:2d} | {c['nombre']:45s} | {c['distrito']:15s} | {c['tipo']}")

In [ ]:
# Filtrar por distrito
centros_chilecito = dao.listar_centros(distrito="Chilecito")

print("=== Centros en Chilecito ===")
for c in centros_chilecito:
    print(f"  {c['nombre']} - {c['direccion']}")

## 5. Medicos

In [ ]:
# Buscar medicos por especialidad
medicos_cardio = dao.buscar_medicos_por_especialidad("Cardiologia")

print("=== Medicos de Cardiologia ===")
for m in medicos_cardio:
    print(f"  {m['nombre']} | Mat: {m['matricula']} | {m['centro']} ({m['distrito']})")

In [ ]:
# Listar medicos por centro
medicos_hospital = dao.listar_medicos_por_centro(1)  # Hospital

print("=== Medicos del Hospital ===")
for m in medicos_hospital:
    print(f"  {m['nombre']} | {m['especialidad']} | {m['matricula']}")

## 6. Pacientes

In [ ]:
pacientes = dao.listar_pacientes()

print("=== Pacientes ===")
for p in pacientes:
    print(f"  ID {p['id_paciente']:2d} | {p['nombre']:25s} | DNI {p['dni']} | {p['distrito']} | {p['obra_social']}")

In [ ]:
# Buscar paciente por DNI
paciente = dao.obtener_paciente_por_dni("40111222")

if paciente:
    print(f"Paciente encontrado:")
    print(f"  Nombre: {paciente['nombre']}")
    print(f"  DNI: {paciente['dni']}")
    print(f"  Telefono: {paciente['telefono']}")
    print(f"  Obra social: {paciente['obra_social']}")

## 7. Agenda de medicos

In [ ]:
# Ver agenda del medico ID 1
agenda = dao.listar_agenda_por_medico(1)

print("=== Agenda - Medico 1 ===")
for a in agenda:
    print(f"  {a['dia_semana']:12s} | {a['hora_inicio']} - {a['hora_fin']} | Duracion: {a['duracion_minutos']}min | Cupo: {a['cupo_diario']}")

## 8. Turnos

In [ ]:
turnos = dao.listar_turnos_proximos(limite=10)

print("=== Proximos Turnos ===")
for t in turnos:
    print(f"  ID {t['id_turno']:3d} | {t['fecha_turno']} | {t['estado']:12s} | ${t['precio_consulta']:>10,.2f}")
    print(f"          Paciente: {t['paciente']} (DNI {t['dni']})")
    print(f"          Medico: {t['medico']} | Centro: {t['centro']} ({t['distrito']})")
    print()

## 9. Disponibilidad por medico

In [ ]:
# Ver disponibilidad del medico 1
disponibilidad = dao.disponibilidad_por_medico(1)

print("=== Disponibilidad - Medico 1 ===")
for d in disponibilidad:
    cupos_libres = d['cupo_diario'] - d['turnos_tomados']
    print(f"  {d['dia_semana']:12s} | {d['hora_inicio']}-{d['hora_fin']} | Cupos: {d['cupo_diario']} | Tomados: {d['turnos_tomados']} | Libres: {cupos_libres}")

## 10. Historial clinico

In [ ]:
# Historial del paciente 1
historial = dao.listar_historial_por_paciente(1)

print("=== Historial Clinico - Paciente 1 ===")
for h in historial:
    print(f"  ID {h['id_historial']:3d} | {h['fecha_registro']} | {h['profesional']}")
    print(f"          Diagnostico: {h['diagnostico']}")
    print(f"          Indicaciones: {h['indicaciones']}")
    print()

## 11. Precio estimado

In [ ]:
# Precio base en Hospital (publico) vs Clinica (privada)
precio_hospital = dao.calcular_precio_estimado(1, 1)  # Centro 1, Especialidad 1
precio_clinica = dao.calcular_precio_estimado(2, 1)   # Centro 2, Especialidad 1

print(f"=== Precio Estimado ===")
print(f"  Hospital (publico):   ${precio_hospital:>10,.2f}")
print(f"  Clinica San Nicolas:  ${precio_clinica:>10,.2f}")

## 12. Operaciones CRUD de ejemplo

In [ ]:
# CREAR paciente de prueba
try:
    dao.crear_paciente(
        dni="99999999",
        nombre="Paciente de Prueba",
        telefono="3825-000000",
        email="test@test.com",
        obra_social="Particular",
        distrito="Chilecito"
    )
    print("Paciente de prueba creado OK")
    
    # Verificar que se creo
    nuevo = dao.obtener_paciente_por_dni("99999999")
    print(f"  Verificado: {nuevo['nombre']} (ID {nuevo['id_paciente']})")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# ACTUALIZAR contacto del paciente
try:
    dao.actualizar_contacto_paciente(1, "3825-111111", "nuevo@email.com")
    print("Contacto actualizado OK")
    
    p = dao.obtener_paciente_por_dni("40111222")
    print(f"  Nuevo telefono: {p['telefono']}")
except Exception as e:
    print(f"Error: {e}")

## 13. Resumen de tablas Oracle

In [ ]:
# Consulta directa de estructura de tablas
query = """
SELECT table_name FROM user_tables 
WHERE table_name NOT LIKE 'BIN$%' 
ORDER BY table_name
"""

tablas = dao.fetch_all(query)
print("=== Tablas del esquema SALUD ===")
for t in tablas:
    print(f"  - {t['table_name'].lower()}")

In [ ]:
# Consulta de columnas por tabla
tabla_buscar = "TURNO"
query_cols = """
SELECT column_name, data_type, nullable
FROM user_tab_columns
WHERE table_name = :tabla
ORDER BY column_id
"""

columnas = dao.fetch_all(query_cols, {"tabla": tabla_buscar})
print(f"=== Columnas de {tabla_buscar} ===")
for c in columnas:
    null_ok = "NULL" if c["nullable"] == "Y" else "NOT NULL"
    print(f"  {c['column_name']:25s} {c['data_type']:20s} {null_ok}")

In [ ]:
# Consulta de constraints (restricciones)
query_constraints = """
SELECT constraint_name, constraint_type, table_name
FROM user_constraints
WHERE table_name NOT LIKE 'BIN$%'
ORDER BY table_name, constraint_type
"""

constraints = dao.fetch_all(query_constraints)
print("=== Restricciones (Constraints) ===")
for c in constraints:
    tipo = {"P": "PK", "R": "FK", "U": "UNIQUE", "C": "CHECK", "V": "VIEW"}.get(c["constraint_type"], c["constraint_type"])
    print(f"  {tipo:8s} | {c['constraint_name']:35s} | {c['table_name']}")

In [ ]:
# Consulta de indices
query_indexes = """
SELECT index_name, table_name, uniqueness
FROM user_indexes
WHERE index_name NOT LIKE 'BIN$%'
ORDER BY table_name, index_name
"""

indices = dao.fetch_all(query_indexes)
print("=== Indices ===")
for i in indices:
    tipo = "UNIQUE" if i["uniqueness"] == "UNIQUE" else "NORMAL"
    print(f"  {tipo:8s} | {i['index_name']:35s} | {i['table_name']}")

## 14. Cerrar conexion

In [ ]:
dao.cerrar()
print("Conexion cerrada OK")